In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup

def find_page(isbn):
    url = r"https://www.aladin.co.kr/search/wsearchresult.aspx?SearchTarget=All&SearchWord={}"
    former_page = requests.get(url.format(isbn))
    fpage_soup = BeautifulSoup(former_page.text, 'html.parser')
    sp_link = fpage_soup.find('a', attrs={'class':"bo3"})

    url = sp_link['href']
    if url.startswith('javascript'):
        real_url = fpage_soup.select_one('a[href*="shop/wproduct.aspx"]')
        if real_url is None:
            return None
        url = real_url['href']
    sp_page = requests.get(url)
    spage_soup = BeautifulSoup(sp_page.text, 'html.parser')
    sp_page = spage_soup.find('div', attrs={'class':'Ere_prod_mconts_box'})
    sp_page_list = sp_page.find_all('ul')
    
    li_list = []
    for ul in sp_page_list:
        li = ul.find_all('li')
        li_list.extend(li)
        
    for i in li_list:
        page = i.get_text(strip = True)
        if '쪽' in page:
            return page
        
        
#데이터프레임 생성        
df_2030 = pd.read_json(r"C:\data\2030s_bestbook_df.json")
df_2030 = df_2030[['no', 'bookname', 'authors', 'publisher', 'publication_year', 'isbn13', 'loan_count']]
page_2030 = df_2030.apply(lambda row: find_page(row['isbn13']), axis = 1 )
page_2030.name = "page_count"
df_2030_with_page = pd.merge(df_2030, page_2030, left_index=True, right_index=True)

page_list = []
for page in df_2030_with_page['page_count']:
    try:
        page_num = int(str(page).replace('쪽', ''))
        page_list.append(page_num)
    except (AttributeError, ValueError, TypeError)as e:
        print(e)
        continue

page_avg = sum(page_list)/len(page_list)
print(page_avg)

# for idx, page in df_2030_with_page['page_count'].items():
#     try:
#         page_nums.append(int(str(page).replace('쪽', '')))
#     except (AttributeError, ValueError, TypeError) as e:
#         errors.append((idx, page, type(e).__name__))

# print("정상 개수:", len(page_nums))
# print("에러 개수:", len(errors))
# print("에러 샘플 5개:", errors[:5])





invalid literal for int() with base 10: 'None'
269.90954773869345


In [ ]:
import gdown
import pandas as pd
import numpy as np

gdown.download(r'https://bit.ly/3GisL6J', 'ns_book4.csv', quiet=False)
ns_book = pd.read_csv('ns_book4.csv', encoding='utf-8', low_memory=False)

# ns_book[ns_book['발행년도'].gt(4000)] = ns_book[ns_book['발행년도'].gt(4000)] - 2333

# def data_fixing(ns_book):
#     #도서권수, 대출건수 int 64로 바꾸기
#     ns_book = ns_book.astype({'도서권수':'int64', '대출건수':'int64'})
    
#     #NaN인 세트 ISBN 공백으로
#     ns_book.fillna({'발행년도':''}, inplace=True)
    
#     setISBN_row = ns_book['세트 ISBN'].isna()
#     ns_book.loc[setISBN_row, '세트 ISBN'] = ''
    
#     #발행년도 4자리로 찾기, 나머지는 -1로 바꾸기
#     ns_book = ns_book.replace({'발행년도':{r'.*(\d{4}).*': r'\1'}}, regex=True)
#     year_str_row = ns_book['발행년도'].astype(str).str.contains(r'\D', na=True)
#     ns_book.loc[year_str_row, '발행년도'] = '-1'
    
#     ns_book = ns_book.astype({'발행년도':'int32'})
    
#     ns_book[ns_book['발행년도'].gt(4000)] = ns_book[ns_book['발행년도'].gt(4000)] - 2333


    #문제되는 데이터 개수 찾기
    
    
    #데이터 누락된 정보 채우기
    
    #남은 결과 삭제하기

Downloading...
From: https://bit.ly/3GisL6J
To: c:\data\ns_book4.csv
100%|██████████| 55.5M/55.5M [00:05<00:00, 9.92MB/s]


In [32]:
ns_book.replace({'발행년도': {r'.*(\d{4}).*': r'\1'}}, regex=True, inplace=True)

na_row = ns_book['발행년도'].str.contains(r'\D', na=True)
ns_book.loc[na_row, '발행년도'] = '-1'

na_row = ns_book['발행년도'].str.contains(r'\D', na=True)
ns_book = ns_book.astype({'발행년도':'int64'})


AttributeError: Can only use .str accessor with string values!

In [36]:
old_years_row = ns_book['발행년도'].gt(0) & ns_book['발행년도'].lt(1900)
ns_book = ns_book[ns_book['발행년도'] != -1]
ns_book[ns_book['발행년도'].eq(-1)]

,번호,도서명,저자,출판사,발행년도,ISBN,세트 ISBN,부가기호,권,주제분류번호,도서권수,대출건수,등록일자
